# Generative Geometric Icons — A Conditional VAE with Cross-Attention and AdaIN Style Transfer

> **Author:** Mohamed El-sadek
> **Project:** Building a Simple Text-to-Image Model from Scratch
> **Notebook:** `final_project.ipynb` — single end-to-end notebook (Colab / Kaggle GPU ready)
> **License:** MIT

---

This notebook builds a **complete, research-grade generative pipeline** for synthesising compositional geometric icons (e.g. *"a red circle inside a blue square"*).
It is designed to run **top-to-bottom** on a free Colab / Kaggle GPU and includes:

- A **synthetic dataset** with thousands of compositional icons (procedurally generated).
- A **Conditional Variational Autoencoder** (Cond-VAE) with an attention-augmented decoder.
- A **Cross-Attention module** that fuses geometry tokens with image features (icon-to-icon attention).
- An **Adaptive Instance Normalisation (AdaIN) Style Transfer** module that recombines the geometry of one icon with the chrominance/style of another.
- A **Gradio application** with six interactive tabs (random generation, composition, style transfer, attention viz, latent explorer, gallery).


## 2. Project Overview

A *compositional* generative model has to do three things at once:

1. **Decode latent variables** into coherent images.
2. **Bind labels** (shape / colour / composition) to the right pixels — i.e. *conditional generation*.
3. **Disentangle structure from appearance**, so we can recombine them — i.e. *style transfer*.

We tackle this with a single model — a Conditional VAE — and two add-on modules: a **Cross-Attention block** that aligns label tokens with feature maps (so the decoder learns *where to put what*), and an **AdaIN block** that swaps feature statistics at inference time to perform *zero-shot* style transfer between icons.


## 3. Problem Statement

Given a structured/textual specification of a geometric composition — e.g.
> *"a red circle inside a blue square"*

— we want a model that can:

- generate a 64×64 RGB image faithful to the specification,
- vary independently along *content* (shape) and *style* (colour / texture),
- expose its internal **attention** so we can explain *how* the geometry attends to the prompt.

The challenge in miniature mirrors that of large-scale text-to-image systems: **compositional grounding** in a low-data regime.


## 4. Objectives

| # | Objective | Deliverable |
|---|---|---|
| O1 | Procedurally generate a labelled icon dataset | `GeometricIconDataset` |
| O2 | Train a Conditional VAE | checkpoint `vae_best.pt` |
| O3 | Implement cross-attention between condition tokens and feature maps | `CrossAttention` module |
| O4 | Implement AdaIN style transfer | `AdaIN`, `style_transfer()` |
| O5 | Provide ≥ 5 generated samples | `samples/final_samples.png` |
| O6 | Build an interactive Gradio app | 6-tab Gradio interface |
| O7 | Produce a 1–2 page brief | `BRIEF.md` |


## 5. Why This Project Matters

- **Pedagogy.** Geometric icons are a *minimal world* in which the dynamics of compositional generation can be studied without a 24-GB VRAM budget.
- **Interpretability.** Because the data is procedurally generated, every image has a known ground-truth caption — we can quantitatively evaluate fidelity and study attention.
- **Transferability.** The cross-attention and AdaIN modules implemented here are the same primitives used in Stable Diffusion-style models; the lessons port directly to large-scale systems.


## 6. Environment Setup — Install Dependencies

We pin a minimal dependency set. The cell auto-installs anything missing on Colab/Kaggle.

In [ ]:
# Auto-install missing packages and ensure a torch build that supports the
# attached GPU. Kaggle currently preinstalls torch 2.10+cu128, which DROPPED
# Pascal (sm_60 / P100) kernel support — calls to cuDNN/GroupNorm fail with
# `cudaErrorNoKernelImageForDevice`. To stay robust on any cluster (P100/T4/A100),
# we sniff `nvidia-smi` BEFORE importing torch and downgrade if needed.

import importlib, subprocess, sys, os

GPU_NAME = ""
try:
    GPU_NAME = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        text=True, stderr=subprocess.DEVNULL,
    ).strip().lower()
except Exception:
    pass
print("nvidia-smi reports GPU:", GPU_NAME or "(none)")

NEEDS_PASCAL_TORCH = ("p100" in GPU_NAME)         # sm_60 — needs the cu121 build (last to ship Pascal kernels)
if NEEDS_PASCAL_TORCH:
    print("Pascal (sm_60) GPU detected — replacing torch with the 2.5.1+cu121 build (supports P100) ...")
    # IMPORTANT: --no-deps so we don't accidentally downgrade numpy (Kaggle ships numpy 2.x;
    # downgrading breaks ABI with the rest of the preinstalled compiled extensions).
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
        "--upgrade", "--force-reinstall", "--no-deps",
        "torch==2.5.1", "torchvision==0.20.1",
        "--index-url", "https://download.pytorch.org/whl/cu121"])

REQUIRED = {
    "torch": "torch",
    "torchvision": "torchvision",
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "tqdm": "tqdm",
    "PIL": "Pillow",
    "sklearn": "scikit-learn",
    "gradio": "gradio",
}

def ensure(pkg_import, pkg_pip):
    try:
        importlib.import_module(pkg_import)
    except ImportError:
        print(f"Installing {pkg_pip} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg_pip])

for imp, pip_name in REQUIRED.items():
    ensure(imp, pip_name)

print("All dependencies satisfied.")

## 7. Imports

In [ ]:
# Standard library
import os, sys, json, math, random, time, warnings
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import List, Tuple, Optional

# Numerics & data
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFilter

# Torch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

# Vision / utilities
import torchvision
from torchvision import transforms as T
from torchvision.utils import make_grid, save_image

# Plotting
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Misc
from tqdm.auto import tqdm
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 140
plt.rcParams["font.family"] = "DejaVu Sans"

print("torch:", torch.__version__, "| torchvision:", torchvision.__version__)


## 8. Reproducibility Setup

Fix all relevant RNG sources so runs are deterministic up to cuDNN non-determinism.

In [ ]:
SEED = 1337

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False  # keep cuDNN fast
    torch.backends.cudnn.benchmark = True

set_seed(SEED)
print(f"Global seed set to {SEED}")


## 9. GPU Setup — Force GPU If Available

We detect CUDA, print VRAM, and enable AMP-friendly settings.

In [ ]:
def gpu_report():
    if not torch.cuda.is_available():
        print("CUDA not available — falling back to CPU.")
        return torch.device("cpu")
    dev = torch.device("cuda:0")
    name = torch.cuda.get_device_name(0)
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    free, _ = torch.cuda.mem_get_info(0)
    free_gb = free / 1024**3
    cc = torch.cuda.get_device_capability(0)
    print(f"GPU            : {name}")
    print(f"Compute cap.   : {cc[0]}.{cc[1]}")
    print(f"Total VRAM     : {total:6.2f} GB")
    print(f"Free VRAM      : {free_gb:6.2f} GB")
    print(f"BF16 supported : {torch.cuda.is_bf16_supported()}")
    return dev

DEVICE  = gpu_report()
USE_AMP = DEVICE.type == "cuda"
print(f"Using device   : {DEVICE} | AMP: {USE_AMP}")


## 10. Project Directories

We create a clean output layout that works identically on Colab, Kaggle and local machines.

In [ ]:
def project_root() -> Path:
    if Path("/kaggle/working").exists():
        return Path("/kaggle/working")
    if Path("/content").exists():
        return Path("/content")
    return Path.cwd()

ROOT = project_root()
DIRS = {
    "outputs":     ROOT / "outputs",
    "checkpoints": ROOT / "checkpoints",
    "figures":     ROOT / "figures",
    "samples":     ROOT / "samples",
    "app":         ROOT / "app",
    "results":     ROOT / "results",
    "data":        ROOT / "data",
}
for p in DIRS.values():
    p.mkdir(parents=True, exist_ok=True)
print("Project root:", ROOT)
for k, v in DIRS.items():
    print(f"  {k:<12} -> {v}")


## 11. Dataset Description

Public icon datasets with structured *composition* labels are scarce, so we **procedurally generate** our own.
Each image is **64 × 64 px RGB** and is built from one of two compositional templates:

| Template | Content |
|---|---|
| `single` | one shape, one colour, on a coloured canvas |
| `nested` | an *inner* shape inside an *outer* shape, each with its own colour |

Variation factors:

- **Shapes (6):** circle, square, triangle, star, pentagon, hexagon
- **Colours (8):** red, blue, green, yellow, purple, orange, cyan, magenta
- **Backgrounds (5):** white, light-grey, beige, navy, black
- **Rotation:** uniform in `[0, 2π)`
- **Position jitter:** ±4 px around centre

Each sample carries a structured label (`shape_outer, color_outer, shape_inner, color_inner, background, composition`) and a natural-language caption (e.g. *"a red circle inside a blue square on a beige background"*).

> A Hugging Face fallback note: we tried browsing the Hub for compositional icon datasets and found nothing with the *structured* (outer/inner shape × colour) labels we need. Procedural generation is therefore the correct choice — it gives us perfect ground-truth captions and an effectively unbounded dataset size.


## 12. Synthetic Dataset Generation

We define a small icon renderer with PIL and a `GeometricIconDataset` that procedurally yields labelled samples.

In [ ]:
IMG_SIZE   = 64
SHAPES     = ["circle", "square", "triangle", "star", "pentagon", "hexagon"]
COLORS     = {
    "red":     (220,  60,  60),
    "blue":    ( 60, 120, 220),
    "green":   ( 60, 180,  90),
    "yellow":  (240, 210,  60),
    "purple":  (160,  80, 200),
    "orange":  (240, 140,  60),
    "cyan":    ( 60, 200, 220),
    "magenta": (220,  80, 200),
}
COLOR_NAMES = list(COLORS.keys())
BACKGROUNDS = {
    "white":     (245, 245, 245),
    "lightgrey": (210, 210, 215),
    "beige":     (235, 225, 200),
    "navy":      ( 25,  35,  70),
    "black":     ( 15,  15,  15),
}
BG_NAMES = list(BACKGROUNDS.keys())
COMPOSITIONS = ["single", "nested"]

# index maps (used as conditional inputs to the model)
SHAPE2IDX = {s: i for i, s in enumerate(SHAPES)}
COLOR2IDX = {c: i for i, c in enumerate(COLOR_NAMES)}
BG2IDX    = {b: i for i, b in enumerate(BG_NAMES)}
COMP2IDX  = {c: i for i, c in enumerate(COMPOSITIONS)}
N_SHAPE, N_COLOR, N_BG, N_COMP = len(SHAPES), len(COLOR_NAMES), len(BG_NAMES), len(COMPOSITIONS)

def regular_polygon(cx, cy, r, n, rot=0.0):
    return [
        (
            cx + r * math.cos(rot + 2 * math.pi * k / n - math.pi / 2),
            cy + r * math.sin(rot + 2 * math.pi * k / n - math.pi / 2),
        )
        for k in range(n)
    ]

def star_polygon(cx, cy, r_out, r_in, points=5, rot=0.0):
    pts = []
    for k in range(points * 2):
        r = r_out if k % 2 == 0 else r_in
        a = rot + math.pi * k / points - math.pi / 2
        pts.append((cx + r * math.cos(a), cy + r * math.sin(a)))
    return pts

def draw_shape(draw, shape, color, cx, cy, r, rot=0.0):
    if shape == "circle":
        draw.ellipse([cx - r, cy - r, cx + r, cy + r], fill=color)
    elif shape == "square":
        # rotated square (regular 4-gon)
        draw.polygon(regular_polygon(cx, cy, r, 4, rot + math.pi / 4), fill=color)
    elif shape == "triangle":
        draw.polygon(regular_polygon(cx, cy, r, 3, rot), fill=color)
    elif shape == "pentagon":
        draw.polygon(regular_polygon(cx, cy, r, 5, rot), fill=color)
    elif shape == "hexagon":
        draw.polygon(regular_polygon(cx, cy, r, 6, rot), fill=color)
    elif shape == "star":
        draw.polygon(star_polygon(cx, cy, r, r * 0.45, points=5, rot=rot), fill=color)
    else:
        raise ValueError(shape)


def render_icon(shape_o, color_o, bg, composition="single",
                shape_i=None, color_i=None, size=IMG_SIZE, jitter=4, rotation=None):
    img = Image.new("RGB", (size, size), BACKGROUNDS[bg])
    draw = ImageDraw.Draw(img)
    cx, cy = size / 2, size / 2
    cx += random.uniform(-jitter, jitter); cy += random.uniform(-jitter, jitter)
    rot_o = rotation if rotation is not None else random.uniform(0, 2 * math.pi)
    r_outer = size * 0.40
    draw_shape(draw, shape_o, COLORS[color_o], cx, cy, r_outer, rot_o)
    if composition == "nested":
        # inner rotation pinned for shape consistency
        rot_i = 0.0
        r_inner = size * 0.18
        draw_shape(draw, shape_i, COLORS[color_i], cx, cy, r_inner, rot_i)
    img = img.filter(ImageFilter.GaussianBlur(radius=0.6))   # mild anti-aliasing
    return img


def random_sample(rng: random.Random) -> dict:
    composition = rng.choice(COMPOSITIONS)
    shape_o = rng.choice(SHAPES)
    color_o = rng.choice(COLOR_NAMES)
    bg      = rng.choice(BG_NAMES)
    if composition == "nested":
        shape_i = rng.choice(SHAPES)
        color_i = rng.choice([c for c in COLOR_NAMES if c != color_o])
    else:
        shape_i, color_i = shape_o, color_o
    cap = (
        f"a {color_o} {shape_o} on a {bg} background"
        if composition == "single"
        else f"a {color_i} {shape_i} inside a {color_o} {shape_o} on a {bg} background"
    )
    return dict(
        shape_outer=shape_o, color_outer=color_o,
        shape_inner=shape_i, color_inner=color_i,
        background=bg, composition=composition, caption=cap,
    )

print("Renderer ready. Example caption:", random_sample(random.Random(0))["caption"])


## 13. Exploratory Data Analysis — Class Distribution

In [ ]:
# Build a 2,000-sample preview to study marginal class distributions.
N_PREVIEW = 2000
rng = random.Random(SEED)
preview = pd.DataFrame([random_sample(rng) for _ in range(N_PREVIEW)])
print(preview.head())
print("\nMarginal counts (head only):")
for col in ["shape_outer", "color_outer", "background", "composition"]:
    print(f"\n{col}:\n{preview[col].value_counts()}")


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.4))
for ax, col in zip(axes, ["shape_outer", "color_outer", "background", "composition"]):
    preview[col].value_counts().plot(kind="bar", ax=ax, color="#4c72b0", edgecolor="black")
    ax.set_title(col); ax.set_xlabel(""); ax.tick_params(axis="x", rotation=30)
plt.suptitle("Marginal class distributions (2,000-sample preview)", y=1.04, fontsize=14)
plt.tight_layout()
plt.savefig(DIRS["figures"] / "eda_marginals.png", bbox_inches="tight")
plt.show()


## 14. Data Visualization — Render a Grid of Samples

In [ ]:
rng = random.Random(SEED + 1)
fig, axes = plt.subplots(4, 8, figsize=(13, 7))
for ax in axes.flat:
    s = random_sample(rng)
    img = render_icon(s["shape_outer"], s["color_outer"], s["background"],
                      composition=s["composition"], shape_i=s["shape_inner"], color_i=s["color_inner"],
                      rotation=0.0)
    ax.imshow(img); ax.axis("off")
    ax.set_title(s["caption"], fontsize=6)
plt.suptitle("Samples from the procedural Icon dataset", y=1.02, fontsize=14)
plt.tight_layout()
plt.savefig(DIRS["figures"] / "sample_grid.png", bbox_inches="tight")
plt.show()


## 15. Data Preprocessing — `GeometricIconDataset`

The dataset object stores the *recipe* for each sample (a dict of label fields) and renders the image **on the fly**. This keeps memory tiny and gives unlimited augmentation diversity.

In [ ]:
TOTAL_SAMPLES = 6000
rng = random.Random(SEED + 2)
ALL_RECIPES = [random_sample(rng) for _ in range(TOTAL_SAMPLES)]

# Tensor transforms — always normalise to [-1, 1] so we can use tanh in decoder
to_tensor = T.Compose([
    T.ToTensor(),                                # [0,1]
    T.Normalize(mean=[0.5]*3, std=[0.5]*3),       # [-1,1]
])

class GeometricIconDataset(Dataset):
    def __init__(self, recipes, augment=False):
        self.recipes = recipes
        self.augment = augment
    def __len__(self):
        return len(self.recipes)
    def __getitem__(self, idx):
        r = self.recipes[idx]
        # rotation pinned to 0 — random rotations cause every regular polygon
        # to look like a rotation-averaged circle to the decoder.
        img = render_icon(
            r["shape_outer"], r["color_outer"], r["background"],
            composition=r["composition"], shape_i=r["shape_inner"], color_i=r["color_inner"],
            rotation=0.0,
        )
        if self.augment and random.random() < 0.5:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
        x = to_tensor(img)
        cond = torch.tensor([
            SHAPE2IDX[r["shape_outer"]], COLOR2IDX[r["color_outer"]],
            SHAPE2IDX[r["shape_inner"]], COLOR2IDX[r["color_inner"]],
            BG2IDX[r["background"]],     COMP2IDX[r["composition"]],
        ], dtype=torch.long)
        return x, cond, r["caption"]

# 80/10/10 split
n_total = len(ALL_RECIPES)
n_train = int(0.8 * n_total); n_val = int(0.1 * n_total)
train_recipes = ALL_RECIPES[:n_train]
val_recipes   = ALL_RECIPES[n_train:n_train + n_val]
test_recipes  = ALL_RECIPES[n_train + n_val:]

train_ds = GeometricIconDataset(train_recipes, augment=True)
val_ds   = GeometricIconDataset(val_recipes,   augment=False)
test_ds  = GeometricIconDataset(test_recipes,  augment=False)

print(f"train: {len(train_ds):>5} | val: {len(val_ds):>5} | test: {len(test_ds):>5}")


## 16. Data Augmentation & DataLoaders

Augmentation is intentionally light: a horizontal flip with p=0.5 (icons are mostly symmetric). *Position and rotation jitter is baked into the renderer* itself, which gives unbounded augmentation for free without ever overfitting a static image.

In [ ]:
BATCH_SIZE  = 128
NUM_WORKERS = 2 if DEVICE.type == "cuda" else 0
PIN         = DEVICE.type == "cuda"

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=PIN, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN)

# sanity check
xb, cb, captions = next(iter(train_loader))
print("image batch  :", tuple(xb.shape), xb.dtype, f"min={xb.min():.2f}, max={xb.max():.2f}")
print("cond batch   :", tuple(cb.shape), cb.dtype)
print("caption sample:", captions[0])


## 17. Model Architecture

We use a **Conditional β-VAE** with three notable design choices:

1. The decoder receives a *learned token sequence* built from condition labels (shape-outer, colour-outer, shape-inner, colour-inner, background, composition).
2. A **cross-attention block** at the 16×16 resolution lets every spatial location query the condition tokens — this is the same pattern as in Stable Diffusion's UNet.
3. An **AdaIN block** is exposed at inference time to perform *content × style* recombination.

```
Image (3×64×64)
   │
   ▼
Encoder (Conv ↓ x4)  ──► flatten ──► [μ, logσ²]  ──► z ∈ ℝ¹²⁸
                                                       │
                                  Condition labels (6) ─►  Embed → tokens (B, 6, 64)
                                                       │
                                                       ▼
                                  Decoder (Linear → 4×4 → ↑x4)
                                       │
                                       │   ┌────── 16×16 features ──┐
                                       │   │  Cross-Attention(q=feat,
                                       │   │  k,v=cond_tokens)
                                       │   └────────────────────────┘
                                       ▼
                                Image (3×64×64)
```


## 18. Cross-Attention Module

Cross-attention computes

$$
\text{Attention}(Q, K, V) = \text{softmax}\!\Big(\frac{QK^\top}{\sqrt{d_k}}\Big)V,
$$

where the **queries Q** come from the spatial feature map (one token per spatial position) and the **keys/values K, V** come from the condition embedding sequence. The result is a feature map where every pixel has been "informed" by the most relevant condition tokens.

In [ ]:
class CrossAttention(nn.Module):
    """Multi-head cross-attention: spatial features attend to a condition token sequence."""
    def __init__(self, dim: int, ctx_dim: int, num_heads: int = 4, dropout: float = 0.0):
        super().__init__()
        assert dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim  = dim // num_heads
        self.scale     = self.head_dim ** -0.5
        self.to_q = nn.Linear(dim,     dim, bias=False)
        self.to_k = nn.Linear(ctx_dim, dim, bias=False)
        self.to_v = nn.Linear(ctx_dim, dim, bias=False)
        self.proj = nn.Linear(dim, dim)
        self.norm_x   = nn.LayerNorm(dim)
        self.norm_ctx = nn.LayerNorm(ctx_dim)
        self.dropout  = nn.Dropout(dropout)
        self.last_attn = None     # cached attention map for visualisation

    def forward(self, x: torch.Tensor, context: torch.Tensor) -> torch.Tensor:
        # x:        (B, C, H, W)
        # context:  (B, T, Cctx)
        B, C, H, W = x.shape
        x_seq = x.flatten(2).transpose(1, 2)             # (B, HW, C)
        x_n   = self.norm_x(x_seq)
        c_n   = self.norm_ctx(context)
        q = self.to_q(x_n).view(B, H * W, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.to_k(c_n).view(B, -1,    self.num_heads, self.head_dim).transpose(1, 2)
        v = self.to_v(c_n).view(B, -1,    self.num_heads, self.head_dim).transpose(1, 2)
        attn = (q @ k.transpose(-2, -1)) * self.scale     # (B, h, HW, T)
        attn = attn.softmax(dim=-1)
        self.last_attn = attn.detach()
        out  = attn @ v                                   # (B, h, HW, d)
        out  = out.transpose(1, 2).reshape(B, H * W, C)
        out  = self.dropout(self.proj(out))
        out  = (x_seq + out).transpose(1, 2).reshape(B, C, H, W)
        return out


## 19. AdaIN — Style Transfer Module

Adaptive Instance Normalisation (Huang & Belongie, 2017) replaces the per-channel statistics of a *content* feature map with those of a *style* feature map:

$$
\text{AdaIN}(c, s) = \sigma(s) \cdot \frac{c - \mu(c)}{\sigma(c)} + \mu(s).
$$

Used inside our VAE's bottleneck, this gives us *training-free* style transfer: encode a content icon, encode a style icon, AdaIN their feature maps, and decode.

In [ ]:
class AdaIN(nn.Module):
    eps: float = 1e-5
    @staticmethod
    def _mean_std(feat: torch.Tensor):
        B, C = feat.shape[:2]
        feat_flat = feat.view(B, C, -1)
        mean = feat_flat.mean(dim=2).view(B, C, 1, 1)
        std  = feat_flat.std(dim=2).view(B, C, 1, 1) + 1e-5
        return mean, std

    def forward(self, content: torch.Tensor, style: torch.Tensor) -> torch.Tensor:
        c_mean, c_std = self._mean_std(content)
        s_mean, s_std = self._mean_std(style)
        normed = (content - c_mean) / c_std
        return normed * s_std + s_mean


## 20. Encoder, Decoder, and the Full Conditional VAE

In [ ]:
def conv_bn(in_c, out_c, k=3, s=1, p=1):
    return nn.Sequential(
        nn.Conv2d(in_c, out_c, k, s, p, bias=False),
        nn.GroupNorm(8, out_c),
        nn.SiLU(inplace=True),
    )

class Encoder(nn.Module):
    """Conv stack 64→32→16→8→4, then linear heads for μ and logσ²."""
    def __init__(self, in_c=3, base=64, z_dim=128):
        super().__init__()
        self.stem  = conv_bn(in_c,    base)                                            # 64
        self.down1 = nn.Sequential(conv_bn(base,    base * 2, s=2), conv_bn(base * 2, base * 2))   # 32
        self.down2 = nn.Sequential(conv_bn(base * 2, base * 4, s=2), conv_bn(base * 4, base * 4))   # 16
        self.down3 = nn.Sequential(conv_bn(base * 4, base * 8, s=2), conv_bn(base * 8, base * 8))   # 8
        self.down4 = nn.Sequential(conv_bn(base * 8, base * 8, s=2), conv_bn(base * 8, base * 8))   # 4
        self.flatten = nn.Flatten()
        self.fc_mu     = nn.Linear(base * 8 * 4 * 4, z_dim)
        self.fc_logvar = nn.Linear(base * 8 * 4 * 4, z_dim)

    def features16(self, x):
        """Return the 16×16 feature map (used by AdaIN style transfer)."""
        h = self.stem(x); h = self.down1(h); h = self.down2(h); return h

    def forward(self, x):
        h = self.stem(x); h = self.down1(h); h = self.down2(h)
        h = self.down3(h); h = self.down4(h)
        h = self.flatten(h)
        return self.fc_mu(h), self.fc_logvar(h)


class Decoder(nn.Module):
    """Inverse of Encoder; cross-attention block at 16×16 attending to condition tokens."""
    def __init__(self, z_dim=128, cond_dim=64, base=64):
        super().__init__()
        self.base = base
        self.fc   = nn.Linear(z_dim, base * 8 * 4 * 4)
        self.up1  = nn.Sequential(nn.Upsample(scale_factor=2, mode="nearest"),
                                  conv_bn(base * 8, base * 8))                          # 8
        self.up2  = nn.Sequential(nn.Upsample(scale_factor=2, mode="nearest"),
                                  conv_bn(base * 8, base * 4))                          # 16
        self.cross = CrossAttention(dim=base * 4, ctx_dim=cond_dim, num_heads=4)        # 16
        self.up3  = nn.Sequential(nn.Upsample(scale_factor=2, mode="nearest"),
                                  conv_bn(base * 4, base * 2))                          # 32
        self.up4  = nn.Sequential(nn.Upsample(scale_factor=2, mode="nearest"),
                                  conv_bn(base * 2, base))                              # 64
        self.head = nn.Sequential(nn.Conv2d(base, 3, 3, padding=1), nn.Tanh())

    def features16_from_z(self, z, cond_tokens):
        h = self.fc(z).view(-1, self.base * 8, 4, 4)
        h = self.up1(h); h = self.up2(h)             # 16×16
        h = self.cross(h, cond_tokens)
        return h

    def decode_from_features16(self, h):
        h = self.up3(h); h = self.up4(h); return self.head(h)

    def forward(self, z, cond_tokens):
        return self.decode_from_features16(self.features16_from_z(z, cond_tokens))


class ConditionEmbedder(nn.Module):
    """Embed (shape_o, color_o, shape_i, color_i, bg, composition) → 6 tokens of size cond_dim."""
    def __init__(self, cond_dim=64):
        super().__init__()
        self.shape_o = nn.Embedding(N_SHAPE, cond_dim)
        self.color_o = nn.Embedding(N_COLOR, cond_dim)
        self.shape_i = nn.Embedding(N_SHAPE, cond_dim)
        self.color_i = nn.Embedding(N_COLOR, cond_dim)
        self.bg      = nn.Embedding(N_BG,    cond_dim)
        self.comp    = nn.Embedding(N_COMP,  cond_dim)
        self.pos     = nn.Parameter(torch.randn(1, 6, cond_dim) * 0.02)

    def forward(self, cond):
        toks = torch.stack([
            self.shape_o(cond[:, 0]), self.color_o(cond[:, 1]),
            self.shape_i(cond[:, 2]), self.color_i(cond[:, 3]),
            self.bg     (cond[:, 4]), self.comp   (cond[:, 5]),
        ], dim=1)                                # (B, 6, cond_dim)
        return toks + self.pos


class CondVAE(nn.Module):
    def __init__(self, z_dim=128, cond_dim=64, base=64):
        super().__init__()
        self.z_dim    = z_dim
        self.cond_dim = cond_dim
        self.encoder  = Encoder(z_dim=z_dim, base=base)
        self.decoder  = Decoder(z_dim=z_dim, cond_dim=cond_dim, base=base)
        self.cond_embedder = ConditionEmbedder(cond_dim=cond_dim)
        self.adain = AdaIN()

    @staticmethod
    def reparameterise(mu, logvar):
        std = (0.5 * logvar).exp()
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x, cond):
        mu, logvar = self.encoder(x)
        z          = self.reparameterise(mu, logvar)
        tokens     = self.cond_embedder(cond)
        recon      = self.decoder(z, tokens)
        return recon, mu, logvar

    @torch.no_grad()
    def generate(self, cond, z=None):
        if z is None:
            z = torch.randn(cond.size(0), self.z_dim, device=cond.device)
        return self.decoder(z, self.cond_embedder(cond))

    @torch.no_grad()
    def style_transfer(self, content_x, style_x, cond, alpha=1.0):
        """AdaIN style transfer at the encoder's 16×16 bottleneck.

        After the AdaIN mix, we re-route through the decoder's cross-attention
        with the content's condition tokens so the feature distribution matches
        what the decoder tail (up3/up4/head) was trained on.
        """
        c_feat = self.encoder.features16(content_x)
        s_feat = self.encoder.features16(style_x)
        mixed  = self.adain(c_feat, s_feat)
        mixed  = alpha * mixed + (1 - alpha) * c_feat
        tokens = self.cond_embedder(cond)
        mixed  = self.decoder.cross(mixed, tokens)
        h = self.decoder.up3(mixed); h = self.decoder.up4(h); return self.decoder.head(h)


## 21. Sanity Check — Forward + Backward Pass on a Mini-Batch

In [ ]:
model = CondVAE().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"Parameter count: {n_params/1e6:.2f} M")

xb_d = xb.to(DEVICE); cb_d = cb.to(DEVICE)
recon, mu, logvar = model(xb_d, cb_d)
print("recon :", tuple(recon.shape))
print("mu    :", tuple(mu.shape))
print("logvar:", tuple(logvar.shape))

loss = F.mse_loss(recon, xb_d)
loss.backward()
print(f"sanity loss = {loss.item():.4f}  ✓")
del recon, mu, logvar, loss
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()


## 22. Hyperparameters

We expose every knob in a single `Config` dataclass so the experiment is fully reproducible.

In [ ]:
@dataclass
class Config:
    epochs: int          = 40
    batch_size: int      = 128
    lr: float            = 2e-3
    weight_decay: float  = 1e-4
    beta_kl: float       = 0.1            # β-VAE: β<1 prevents posterior collapse
    beta_warmup_epochs: int = 2
    grad_clip: float     = 1.0
    z_dim: int           = 128
    cond_dim: int        = 64
    early_stop_patience: int = 12
    use_amp: bool        = True
    log_interval: int    = 50

CFG = Config()
print(asdict(CFG))

## 23. Loss Functions

The objective is the **β-VAE ELBO**:

$$
\mathcal{L} = \underbrace{\mathbb{E}_{q_\phi(z|x)}\!\left[\|x - \hat x\|_2^2\right]}_{\text{reconstruction}} \;+\; \beta \cdot \underbrace{D_\text{KL}\!\left(q_\phi(z|x)\,\|\,\mathcal{N}(0, I)\right)}_{\text{regularisation}}.
$$

We additionally include a *KL warm-up*: β ramps from 0 → 1 over the first `beta_warmup_epochs` to stop the decoder collapsing into the prior.

In [ ]:
def vae_loss(recon, target, mu, logvar, beta: float):
    # Mix MSE (smooth) with L1 (sharp edges) — L1 alone is a known fix for VAE blur.
    recon_loss = 0.5 * F.mse_loss(recon, target, reduction="mean") \
               + 0.5 * F.l1_loss(recon, target, reduction="mean")
    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + beta * kl, recon_loss, kl

def beta_schedule(epoch: int, cfg: Config) -> float:
    if cfg.beta_warmup_epochs <= 0:
        return cfg.beta_kl
    return cfg.beta_kl * min(1.0, epoch / cfg.beta_warmup_epochs)


## 24. Optimizer & Scheduler

`AdamW` with cosine learning-rate decay; `GradScaler` for AMP.

In [ ]:
def build_optim(model: nn.Module, cfg: Config, steps_per_epoch: int):
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.epochs * steps_per_epoch)
    scaler = GradScaler(enabled=cfg.use_amp and DEVICE.type == "cuda")
    return opt, sched, scaler


## 25. Training Loop

The loop combines AMP, gradient clipping, KL warm-up, periodic validation, and best-checkpoint saving.

In [ ]:
@torch.no_grad()
def evaluate(model, loader, beta):
    model.eval()
    total = rec_total = kl_total = 0.0
    n = 0
    for x, c, _ in loader:
        x, c = x.to(DEVICE), c.to(DEVICE)
        recon, mu, logvar = model(x, c)
        loss, rec, kl = vae_loss(recon, x, mu, logvar, beta)
        bs = x.size(0)
        total     += loss.item() * bs
        rec_total += rec.item()  * bs
        kl_total  += kl.item()   * bs
        n         += bs
    model.train()
    return total / n, rec_total / n, kl_total / n


def train(model, train_loader, val_loader, cfg: Config, ckpt_name="vae_best.pt"):
    steps_per_epoch = len(train_loader)
    opt, sched, scaler = build_optim(model, cfg, steps_per_epoch)
    history = {"train_loss": [], "train_rec": [], "train_kl": [],
               "val_loss":   [], "val_rec":   [], "val_kl":   [],
               "lr": [], "beta": [], "epoch_time": []}
    best_val = float("inf"); patience = cfg.early_stop_patience
    ckpt_path = DIRS["checkpoints"] / ckpt_name
    model.train()
    last_epoch = 0
    for epoch in range(1, cfg.epochs + 1):
        last_epoch = epoch
        t0 = time.time()
        beta = beta_schedule(epoch, cfg)
        running = {"loss": 0.0, "rec": 0.0, "kl": 0.0, "n": 0}
        pbar = tqdm(train_loader, desc=f"epoch {epoch:02d}/{cfg.epochs} β={beta:.2f}", leave=False)
        for x, c, _ in pbar:
            x, c = x.to(DEVICE, non_blocking=True), c.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with autocast(enabled=cfg.use_amp and DEVICE.type == "cuda"):
                recon, mu, logvar = model(x, c)
                loss, rec, kl = vae_loss(recon, x, mu, logvar, beta)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(opt); scaler.update(); sched.step()
            bs = x.size(0)
            running["loss"] += loss.item() * bs
            running["rec"]  += rec.item()  * bs
            running["kl"]   += kl.item()   * bs
            running["n"]    += bs
            pbar.set_postfix(loss=f"{loss.item():.3f}", rec=f"{rec.item():.3f}", kl=f"{kl.item():.3f}")

        train_loss = running["loss"] / running["n"]
        train_rec  = running["rec"]  / running["n"]
        train_kl   = running["kl"]   / running["n"]
        val_loss, val_rec, val_kl = evaluate(model, val_loader, beta)
        elapsed = time.time() - t0

        history["train_loss"].append(train_loss); history["train_rec"].append(train_rec); history["train_kl"].append(train_kl)
        history["val_loss"].append(val_loss);     history["val_rec"].append(val_rec);     history["val_kl"].append(val_kl)
        history["lr"].append(opt.param_groups[0]["lr"]); history["beta"].append(beta); history["epoch_time"].append(elapsed)

        flag = ""
        if val_loss < best_val - 1e-4:
            best_val = val_loss; patience = cfg.early_stop_patience
            torch.save({"model": model.state_dict(), "cfg": asdict(cfg), "epoch": epoch}, ckpt_path)
            flag = "  ★ saved"
        else:
            patience -= 1
        print(f"epoch {epoch:02d} | train {train_loss:.4f} (rec {train_rec:.4f}, kl {train_kl:.4f}) "
              f"| val {val_loss:.4f} (rec {val_rec:.4f}, kl {val_kl:.4f}) | t {elapsed:5.1f}s{flag}")
        if patience <= 0:
            print(f"early stopping at epoch {epoch} — no improvement for {cfg.early_stop_patience} epochs")
            break

    torch.save({"model": model.state_dict(), "cfg": asdict(cfg), "epoch": last_epoch},
               DIRS["checkpoints"] / ckpt_name.replace("best", "final"))
    return history


## 26. Run Training

In [ ]:
set_seed(SEED)
model = CondVAE(z_dim=CFG.z_dim, cond_dim=CFG.cond_dim).to(DEVICE)
history = train(model, train_loader, val_loader, CFG, ckpt_name="vae_best.pt")
with open(DIRS["results"] / "history.json", "w") as f:
    json.dump(history, f, indent=2)
print("Training complete. Best checkpoint:", DIRS["checkpoints"] / "vae_best.pt")


## 27. Plot Training Curves

In [ ]:
def plot_curves(history, out_path: Path):
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(epochs, history["train_loss"], label="train")
    axes[0].plot(epochs, history["val_loss"],   label="val")
    axes[0].set_title("Total Loss");  axes[0].set_xlabel("epoch"); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(epochs, history["train_rec"], label="train")
    axes[1].plot(epochs, history["val_rec"],   label="val")
    axes[1].set_title("Reconstruction (MSE)"); axes[1].set_xlabel("epoch"); axes[1].legend(); axes[1].grid(alpha=0.3)
    axes[2].plot(epochs, history["train_kl"], label="train")
    axes[2].plot(epochs, history["val_kl"],   label="val")
    axes[2].set_title("KL Divergence"); axes[2].set_xlabel("epoch"); axes[2].legend(); axes[2].grid(alpha=0.3)
    plt.suptitle("Training dynamics", y=1.02, fontsize=14)
    plt.tight_layout(); plt.savefig(out_path, bbox_inches="tight"); plt.show()

plot_curves(history, DIRS["figures"] / "training_curves.png")


## 28. Evaluation Metrics

Because we have *ground-truth* captions for every test image we can evaluate the model on three concrete axes:

| Metric | What it measures |
|---|---|
| **Recon-MSE** | how faithfully the model reconstructs a held-out image |
| **KL** | how Gaussian-prior-aligned the latent is |
| **Conditional Colour Accuracy (CCA)** | given a target colour label, does the dominant non-background colour of the generated image match? |


In [ ]:
# Reload best checkpoint
ckpt = torch.load(DIRS["checkpoints"] / "vae_best.pt", map_location=DEVICE)
model.load_state_dict(ckpt["model"])
model.eval()

# 1) Reconstruction MSE & KL on the test set
test_loss, test_rec, test_kl = evaluate(model, test_loader, beta=CFG.beta_kl)
print(f"TEST  loss={test_loss:.4f} | recon-MSE={test_rec:.4f} | KL={test_kl:.4f}")

# 2) Conditional Colour Accuracy
def dominant_colour(img_t: torch.Tensor, bg_idx: int = -1) -> int:
    """Return the index in COLORS of the colour that dominates the foreground.

    Pixels whose nearest match is the prompted background are masked out so the
    canvas doesn’t out-vote the icon itself.
    """
    img = (img_t.clamp(-1, 1) + 1) / 2
    pixels = img.permute(1, 2, 0).reshape(-1, 3) * 255
    full_palette = list(COLORS.values()) + list(BACKGROUNDS.values())
    palette_t = torch.tensor(full_palette, dtype=pixels.dtype, device=pixels.device)
    nearest  = ((pixels.unsqueeze(1) - palette_t.unsqueeze(0)) ** 2).sum(-1).argmin(-1)
    n_fg = len(COLORS)
    if bg_idx >= 0:
        # mask out pixels that nearest-match the prompted background
        bg_palette_idx = n_fg + bg_idx
        keep = nearest != bg_palette_idx
        nearest = nearest[keep]
    # also drop any pixel that mapped to ANY background colour
    nearest = nearest[nearest < n_fg]
    if nearest.numel() == 0:
        return 0
    counts = torch.bincount(nearest, minlength=n_fg)
    return counts.argmax().item()

correct = 0; total = 0
with torch.no_grad():
    for x, c, _ in test_loader:
        c = c.to(DEVICE)
        gen = model.generate(c)
        for i in range(gen.size(0)):
            pred_color = dominant_colour(gen[i].cpu(), bg_idx=c[i, 4].item())
            true_color = c[i, 1].item()
            correct += int(pred_color == true_color); total += 1
        if total >= 1024:
            break
cca = correct / total
print(f"Conditional Colour Accuracy (outer-colour): {cca*100:5.2f}%  ({correct}/{total})")

metrics = {"test_loss": test_loss, "recon_mse": test_rec, "kl": test_kl,
           "conditional_color_accuracy": cca}
with open(DIRS["results"] / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
metrics


## 29. Latent Space Visualization (PCA / t-SNE)

We embed 1,000 test images into the latent space and project to 2-D.

In [ ]:
@torch.no_grad()
def collect_latents(model, loader, max_n=1000):
    zs, labels = [], []
    n = 0
    for x, c, _ in loader:
        x = x.to(DEVICE)
        mu, _ = model.encoder(x)
        zs.append(mu.cpu()); labels.append(c)
        n += x.size(0)
        if n >= max_n: break
    return torch.cat(zs)[:max_n].numpy(), torch.cat(labels)[:max_n].numpy()

Z, L = collect_latents(model, test_loader, max_n=1000)
print("latent matrix:", Z.shape)

pca  = PCA(n_components=2).fit_transform(Z)
tsne = TSNE(n_components=2, init="pca", learning_rate="auto",
            perplexity=30, random_state=SEED).fit_transform(Z)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, proj, title in zip(axes, [pca, tsne], ["PCA", "t-SNE"]):
    ax.scatter(proj[:, 0], proj[:, 1], c=L[:, 0], cmap="tab10", s=12, alpha=0.85)
    ax.set_title(f"{title} — coloured by outer shape")
    ax.set_xticks([]); ax.set_yticks([])
handles = [mpatches.Patch(color=plt.cm.tab10(i / 10.0), label=SHAPES[i]) for i in range(N_SHAPE)]
fig.legend(handles=handles, loc="lower center", ncol=N_SHAPE, bbox_to_anchor=(0.5, -0.05))
plt.tight_layout(); plt.savefig(DIRS["figures"] / "latent_space.png", bbox_inches="tight"); plt.show()


## 30. Generated Samples — The 5 Required Final Samples

We hand-pick five compositional prompts that span the full design space and render each.

In [ ]:
def encode_caption(shape_o, color_o, shape_i=None, color_i=None, bg="white", composition="single"):
    if shape_i is None: shape_i = shape_o
    if color_i is None: color_i = color_o
    return torch.tensor([[
        SHAPE2IDX[shape_o], COLOR2IDX[color_o],
        SHAPE2IDX[shape_i], COLOR2IDX[color_i],
        BG2IDX[bg],         COMP2IDX[composition],
    ]], dtype=torch.long, device=DEVICE)

def to_pil(t: torch.Tensor) -> Image.Image:
    t = (t.clamp(-1, 1) + 1) / 2
    return Image.fromarray((t.cpu().permute(1, 2, 0).numpy() * 255).astype(np.uint8))

FINAL_PROMPTS = [
    dict(caption="a red circle on a white background",
         shape_o="circle", color_o="red", bg="white", composition="single"),
    dict(caption="a blue star on a black background",
         shape_o="star", color_o="blue", bg="black", composition="single"),
    dict(caption="a yellow circle inside a green square on a beige background",
         shape_o="square", color_o="green", shape_i="circle", color_i="yellow",
         bg="beige", composition="nested"),
    dict(caption="a magenta triangle inside a cyan hexagon on a navy background",
         shape_o="hexagon", color_o="cyan", shape_i="triangle", color_i="magenta",
         bg="navy", composition="nested"),
    dict(caption="an orange star inside a purple pentagon on a lightgrey background",
         shape_o="pentagon", color_o="purple", shape_i="star", color_i="orange",
         bg="lightgrey", composition="nested"),
]

set_seed(SEED + 7)
fig, axes = plt.subplots(1, 5, figsize=(15, 3.4))
for i, p in enumerate(FINAL_PROMPTS):
    cond = encode_caption(**{k: v for k, v in p.items() if k != "caption"})
    with torch.no_grad():
        gen = model.generate(cond)[0]
    pil = to_pil(gen)
    pil.save(DIRS["samples"] / f"final_sample_{i+1:02d}.png")
    axes[i].imshow(pil); axes[i].axis("off")
    axes[i].set_title(p["caption"], fontsize=8, wrap=True)
plt.suptitle("Five final generated samples (Cond-VAE)", y=1.04, fontsize=14)
plt.tight_layout(); plt.savefig(DIRS["samples"] / "final_samples.png", bbox_inches="tight"); plt.show()


In [ ]:
# Bonus: 8x8 random gallery
set_seed(SEED + 11)
gallery_conds = []
for _ in range(64):
    s = random_sample(random.Random(random.randint(0, 1 << 30)))
    gallery_conds.append([
        SHAPE2IDX[s["shape_outer"]], COLOR2IDX[s["color_outer"]],
        SHAPE2IDX[s["shape_inner"]], COLOR2IDX[s["color_inner"]],
        BG2IDX[s["background"]],     COMP2IDX[s["composition"]],
    ])
gallery_conds = torch.tensor(gallery_conds, device=DEVICE)
with torch.no_grad():
    gallery = model.generate(gallery_conds)
grid = make_grid((gallery + 1) / 2, nrow=8, padding=2)
save_image(grid, DIRS["samples"] / "gallery_8x8.png")
plt.figure(figsize=(10, 10))
plt.imshow(grid.cpu().permute(1, 2, 0).numpy())
plt.axis("off"); plt.title("64-image generated gallery", fontsize=14)
plt.savefig(DIRS["figures"] / "gallery_8x8.png", bbox_inches="tight")
plt.show()


## 31. Style Transfer Results — AdaIN on Icon Pairs

For every pair (content, style) we encode both, mix their feature statistics with AdaIN at the 16×16 bottleneck, and decode. We sweep the AdaIN strength `α` from 0 (pure content) to 1 (full style swap).

In [ ]:
pairs = []
test_iter = iter(test_loader)
xb_t, cb_t, _ = next(test_iter)
xb_t = xb_t.to(DEVICE); cb_t = cb_t.to(DEVICE)
for i in range(4):
    pairs.append((xb_t[i], xb_t[i + 8], cb_t[i].unsqueeze(0)))

alphas = [0.0, 0.33, 0.66, 1.0]
fig, axes = plt.subplots(len(pairs), len(alphas) + 2, figsize=(2.0 * (len(alphas) + 2), 2.0 * len(pairs)))
for r, (c_img, s_img, c_cond) in enumerate(pairs):
    axes[r, 0].imshow(to_pil(c_img)); axes[r, 0].axis("off")
    if r == 0: axes[r, 0].set_title("Content")
    axes[r, 1].imshow(to_pil(s_img)); axes[r, 1].axis("off")
    if r == 0: axes[r, 1].set_title("Style")
    for k, a in enumerate(alphas):
        with torch.no_grad():
            mixed = model.style_transfer(c_img.unsqueeze(0), s_img.unsqueeze(0), c_cond, alpha=a)[0]
        axes[r, k + 2].imshow(to_pil(mixed)); axes[r, k + 2].axis("off")
        if r == 0: axes[r, k + 2].set_title(f"α={a:.2f}")
plt.suptitle("AdaIN style transfer between icons", y=1.02, fontsize=14)
plt.tight_layout(); plt.savefig(DIRS["figures"] / "style_transfer.png", bbox_inches="tight"); plt.show()


## 32. Cross-Attention Visualization

For each generated image we average the attention weights over heads and reshape them back to a 16×16 spatial map *per condition token*. Bright regions are the pixels that attended *most* to that token.

In [ ]:
TOKEN_NAMES = ["shape_outer", "color_outer", "shape_inner", "color_inner", "background", "composition"]

@torch.no_grad()
def visualize_attention(prompt: dict, save: Optional[Path] = None):
    cond = encode_caption(**{k: v for k, v in prompt.items() if k != "caption"})
    z = torch.randn(1, model.z_dim, device=DEVICE)
    img = model.decoder(z, model.cond_embedder(cond))
    attn = model.decoder.cross.last_attn         # (1, h, HW, T)
    attn = attn.mean(dim=1)[0]                   # (HW, T)
    H = W = 16
    fig, axes = plt.subplots(1, len(TOKEN_NAMES) + 1, figsize=(2.1 * (len(TOKEN_NAMES) + 1), 2.4))
    axes[0].imshow(to_pil(img[0])); axes[0].set_title("generated"); axes[0].axis("off")
    for t, name in enumerate(TOKEN_NAMES):
        m = attn[:, t].reshape(H, W).cpu().numpy()
        m = (m - m.min()) / (m.max() - m.min() + 1e-9)
        m_up = np.kron(m, np.ones((4, 4)))
        axes[t + 1].imshow(to_pil(img[0]))
        axes[t + 1].imshow(m_up, cmap="inferno", alpha=0.55)
        axes[t + 1].set_title(name, fontsize=8); axes[t + 1].axis("off")
    plt.suptitle(prompt["caption"], y=1.02, fontsize=11)
    plt.tight_layout()
    if save: plt.savefig(save, bbox_inches="tight")
    plt.show()

for i, p in enumerate(FINAL_PROMPTS[:3]):
    visualize_attention(p, save=DIRS["figures"] / f"attention_map_{i+1}.png")


## 33. Ablation Study — Does Cross-Attention Matter?

We train a *no-attention* variant of the decoder with the same training budget and compare conditional colour accuracy. Cross-attention should give a clear lift on *compositional* prompts.

In [ ]:
class DecoderNoAttn(Decoder):
    def __init__(self, z_dim=128, cond_dim=64, base=64):
        super().__init__(z_dim, cond_dim, base)
        self.cross = nn.Identity()
    def features16_from_z(self, z, cond_tokens):
        h = self.fc(z).view(-1, self.base * 8, 4, 4)
        h = self.up1(h); h = self.up2(h)
        return h          # bypass cross-attention

class CondVAENoAttn(CondVAE):
    def __init__(self, z_dim=128, cond_dim=64, base=64):
        super().__init__(z_dim, cond_dim, base)
        self.decoder = DecoderNoAttn(z_dim, cond_dim, base)

ablate_cfg = Config(epochs=6, beta_warmup_epochs=1, log_interval=200, early_stop_patience=20)

print("Training NO-attention baseline ...")
set_seed(SEED)
ablate_model = CondVAENoAttn().to(DEVICE)
_ = train(ablate_model, train_loader, val_loader, ablate_cfg, ckpt_name="vae_noattn_best.pt")

def cca_of(m):
    correct = 0; total = 0
    with torch.no_grad():
        for x, c, _ in test_loader:
            c = c.to(DEVICE)
            gen = m.generate(c)
            for i in range(gen.size(0)):
                correct += int(dominant_colour(gen[i].cpu(), bg_idx=c[i, 4].item()) == c[i, 1].item()); total += 1
            if total >= 1024: break
    return correct / total

cca_noattn = cca_of(ablate_model)

print("Training matched-budget WITH-attention baseline ...")
set_seed(SEED)
match_model = CondVAE().to(DEVICE)
_ = train(match_model, train_loader, val_loader, ablate_cfg, ckpt_name="vae_match_best.pt")
cca_attn = cca_of(match_model)

ablation = {
    "no_attention_CCA":   cca_noattn,
    "with_attention_CCA": cca_attn,
    "delta":              cca_attn - cca_noattn,
}
with open(DIRS["results"] / "ablation.json", "w") as f:
    json.dump(ablation, f, indent=2)
print(json.dumps(ablation, indent=2))


## 34. Error Analysis — Where the Model Fails

We render reconstructions of test samples and surface the worst-MSE cases.

In [ ]:
errs, cap_list, orig_list, recon_list = [], [], [], []
with torch.no_grad():
    for x, c, caps in test_loader:
        x_d = x.to(DEVICE); c_d = c.to(DEVICE)
        recon, _, _ = model(x_d, c_d)
        e = ((recon - x_d) ** 2).mean(dim=(1, 2, 3))
        for i in range(x_d.size(0)):
            errs.append(e[i].item()); cap_list.append(caps[i])
            orig_list.append(x_d[i].cpu()); recon_list.append(recon[i].cpu())
        if len(errs) >= 256: break
errs = np.array(errs)
worst = np.argsort(errs)[-8:][::-1]
fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
for k, idx in enumerate(worst):
    axes[0, k].imshow(to_pil(orig_list[idx]));   axes[0, k].axis("off"); axes[0, k].set_title(f"GT — MSE {errs[idx]:.3f}", fontsize=7)
    axes[1, k].imshow(to_pil(recon_list[idx])); axes[1, k].axis("off"); axes[1, k].set_title(cap_list[idx], fontsize=6, wrap=True)
plt.suptitle("Worst-8 reconstructions (top: ground truth, bottom: reconstruction)", y=1.02, fontsize=13)
plt.tight_layout(); plt.savefig(DIRS["figures"] / "error_analysis.png", bbox_inches="tight"); plt.show()


## 35. Limitations

- **Synthetic domain.** The dataset is procedurally generated — generalisation to *natural* icons (e.g. SVG icon packs, FontAwesome) is untested.
- **Resolution.** 64 × 64 keeps the model snappy on a free GPU but is too low for nuanced texture style transfer.
- **VAE blur.** Conditional VAEs are known to produce mildly blurry samples; a small adversarial term or a diffusion head would sharpen them.
- **Fixed token vocabulary.** Conditions are categorical — there is no genuine *text encoder*. A frozen CLIP text encoder would unlock free-form prompts.


## 36. Future Improvements

1. **Replace the categorical conditioner with a frozen CLIP text encoder** so the model accepts arbitrary prompts.
2. **Switch the decoder to a tiny U-Net diffusion model** trained with v-prediction (Salimans & Ho, 2022). The cross-attention block ports unchanged.
3. **Add LPIPS perceptual loss** on top of MSE for crisper reconstructions.
4. **Disentanglement metrics**: compute a β-TC-VAE-style mutual-information gap on shape vs. colour to quantify factor separation.
5. **Web-deployable export**: convert the encoder/decoder to ONNX so the Gradio app can run client-side.


## 37. Conclusion

We trained a Conditional VAE with a cross-attention decoder and an AdaIN style-transfer head end-to-end on a procedurally generated icon dataset. The model:

- reconstructs held-out icons with **low MSE**,
- conditionally generates icons that respect the **outer-colour** and **outer-shape** prompts (see CCA in §28),
- supports **AdaIN-based style transfer** without any fine-tuning,
- exposes **interpretable cross-attention maps** that align condition tokens with image regions.

The included Gradio app lets a user explore all of these modes interactively.


## 38. References & Harvard Citations

- **Kingma, D.P. & Welling, M.** (2014) *Auto-Encoding Variational Bayes*. arXiv:1312.6114.
- **Sohn, K., Yan, X. & Lee, H.** (2015) *Learning Structured Output Representation Using Deep Conditional Generative Models*. NeurIPS.
- **Higgins, I. et al.** (2017) *β-VAE: Learning Basic Visual Concepts with a Constrained Variational Framework*. ICLR.
- **Vaswani, A. et al.** (2017) *Attention Is All You Need*. NeurIPS.
- **Huang, X. & Belongie, S.** (2017) *Arbitrary Style Transfer in Real-time with Adaptive Instance Normalization*. ICCV.
- **Rombach, R. et al.** (2022) *High-Resolution Image Synthesis with Latent Diffusion Models*. CVPR.
- **Ho, J., Jain, A. & Abbeel, P.** (2020) *Denoising Diffusion Probabilistic Models*. NeurIPS.
- **Salimans, T. & Ho, J.** (2022) *Progressive Distillation for Fast Sampling of Diffusion Models*. ICLR.
- **Radford, A. et al.** (2021) *Learning Transferable Visual Models From Natural Language Supervision (CLIP)*. ICML.
- **Goodfellow, I. et al.** (2014) *Generative Adversarial Nets*. NeurIPS.

### Tool References

- **Paszke, A. et al.** (2019) *PyTorch: An Imperative Style, High-Performance Deep Learning Library*. NeurIPS.
- **Abid, A. et al.** (2019) *Gradio: Hassle-Free Sharing and Testing of ML Models in the Wild*. arXiv:1906.02569.


## 39. Appendix — Reproducibility Card

| Item | Value |
|---|---|
| Image size | 64 × 64 RGB |
| Dataset size | 6,000 icons (80/10/10) |
| Latent size | 128 |
| Cond. embedding size | 64 |
| Conditioning tokens | 6 (shape_o, color_o, shape_i, color_i, bg, composition) |
| Cross-attention heads | 4 |
| Optimizer | AdamW (lr 2e-3, wd 1e-4) |
| Scheduler | CosineAnnealingLR |
| Batch size | 128 |
| Epochs | 30 (early-stop patience 8) |
| Mixed precision | enabled if CUDA |
| Seed | 1337 |


## 40. Gradio Demo App — 6-Tab Interface

The app loads the trained model and exposes:

1. **Random Generation** — pick a random latent and a structured prompt.
2. **Shape Composition** — choose outer / inner shapes & colours.
3. **Style Transfer** — content × style with an `α` slider.
4. **Cross-Attention Visualization** — heatmap per condition token.
5. **Latent Space Explorer** — slide the first eight z dimensions.
6. **Gallery** — generates a fresh 64-image grid every click.

Set `share=True` if you want a public URL (works on Colab/Kaggle).

In [ ]:
import gradio as gr

def _img(t):
    return to_pil(t.detach().cpu())

# ──────────────── Tab 1: Random Generation ────────────────
@torch.no_grad()
def tab_random(seed: int):
    torch.manual_seed(int(seed))
    s = random_sample(random.Random(int(seed)))
    cond = encode_caption(s["shape_outer"], s["color_outer"], s["shape_inner"], s["color_inner"],
                          bg=s["background"], composition=s["composition"])
    img = model.generate(cond)[0]
    return _img(img), s["caption"]

# ──────────────── Tab 2: Shape Composition ────────────────
@torch.no_grad()
def tab_compose(shape_o, color_o, shape_i, color_i, bg, composition, seed):
    torch.manual_seed(int(seed))
    cond = encode_caption(shape_o, color_o, shape_i, color_i, bg=bg, composition=composition)
    return _img(model.generate(cond)[0])

# ──────────────── Tab 3: Style Transfer ────────────────
@torch.no_grad()
def tab_style(c_shape, c_color, c_bg, s_shape, s_color, s_bg, alpha):
    c_pil = render_icon(c_shape, c_color, c_bg, composition="single", rotation=0.0)
    s_pil = render_icon(s_shape, s_color, s_bg, composition="single", rotation=0.0)
    c_t = to_tensor(c_pil).unsqueeze(0).to(DEVICE)
    s_t = to_tensor(s_pil).unsqueeze(0).to(DEVICE)
    cond = encode_caption(c_shape, c_color, bg=c_bg, composition="single")
    out = model.style_transfer(c_t, s_t, cond, alpha=float(alpha))[0]
    return c_pil, s_pil, _img(out)

# ──────────────── Tab 4: Cross-Attention ────────────────
@torch.no_grad()
def tab_attention(shape_o, color_o, shape_i, color_i, bg, composition, seed):
    torch.manual_seed(int(seed))
    cond = encode_caption(shape_o, color_o, shape_i, color_i, bg=bg, composition=composition)
    z = torch.randn(1, model.z_dim, device=DEVICE)
    img = model.decoder(z, model.cond_embedder(cond))[0]
    attn = model.decoder.cross.last_attn.mean(dim=1)[0]
    H = W = 16
    panels = [(_img(img), "generated")]
    base = np.array(_img(img)).astype(np.float32) / 255.0
    for t, name in enumerate(TOKEN_NAMES):
        m = attn[:, t].reshape(H, W).cpu().numpy()
        m = (m - m.min()) / (m.max() - m.min() + 1e-9)
        m_up = np.kron(m, np.ones((4, 4)))
        heat = plt.cm.inferno(m_up)[..., :3]
        blend = (0.45 * heat + 0.55 * base).clip(0, 1)
        panels.append((Image.fromarray((blend * 255).astype(np.uint8)), name))
    return panels

# ──────────────── Tab 5: Latent Explorer ────────────────
@torch.no_grad()
def tab_latent(shape_o, color_o, bg, z0, z1, z2, z3, z4, z5, z6, z7):
    cond = encode_caption(shape_o, color_o, bg=bg, composition="single")
    z = torch.zeros(1, model.z_dim, device=DEVICE)
    for i, v in enumerate([z0, z1, z2, z3, z4, z5, z6, z7]):
        z[0, i] = float(v)
    return _img(model.decoder(z, model.cond_embedder(cond))[0])

# ──────────────── Tab 6: Gallery ────────────────
@torch.no_grad()
def tab_gallery(seed: int):
    torch.manual_seed(int(seed))
    rng = random.Random(int(seed))
    conds = []
    for _ in range(64):
        s = random_sample(rng)
        conds.append([SHAPE2IDX[s["shape_outer"]], COLOR2IDX[s["color_outer"]],
                      SHAPE2IDX[s["shape_inner"]], COLOR2IDX[s["color_inner"]],
                      BG2IDX[s["background"]], COMP2IDX[s["composition"]]])
    conds = torch.tensor(conds, device=DEVICE)
    out = model.generate(conds)
    grid = make_grid((out + 1) / 2, nrow=8, padding=2)
    return Image.fromarray((grid.cpu().permute(1, 2, 0).numpy() * 255).astype(np.uint8))


HEADER = """
# Geometric Icon Generator
**Conditional VAE · Cross-Attention · AdaIN Style Transfer**

Author **Mohamed El-sadek** · Built with PyTorch · Live demo of a 16 M-parameter
Cond-VAE trained on a procedurally generated icon dataset.

Pick a tab to explore:
1. **Random Generation** — sample a fresh icon from a random prompt
2. **Shape Composition** — choose shape × colour × background × inner shape
3. **Style Transfer** — recombine the geometry of one icon with the colour palette of another (AdaIN)
4. **Cross-Attention** — overlay attention heatmaps for each condition token
5. **Latent Explorer** — slide the first 8 z-dimensions
6. **Gallery** — render a fresh 64-icon grid in one click
"""

with gr.Blocks(title="Geometric Icon Generator — Mohamed El-sadek", theme=gr.themes.Soft()) as demo:
    gr.Markdown(HEADER)

    # ───── Tab 1 ─────
    with gr.Tab("🎲 1. Random Generation"):
        gr.Markdown("Re-roll a complete random prompt and sample a latent.")
        with gr.Row():
            seed1 = gr.Slider(0, 9999, value=0, step=1, label="seed")
            btn1  = gr.Button("Generate", variant="primary")
        out1 = gr.Image(label="generated icon", show_download_button=True, height=320)
        cap1 = gr.Textbox(label="prompt the model received", interactive=False)
        btn1.click(tab_random, seed1, [out1, cap1])
        gr.Examples([[0], [7], [42], [123], [777]], inputs=seed1, label="Example seeds")

    # ───── Tab 2 ─────
    with gr.Tab("🧩 2. Shape Composition"):
        gr.Markdown("Hand-design an icon: outer shape, outer colour, inner shape, inner colour, background, composition.")
        with gr.Row():
            so  = gr.Dropdown(SHAPES,       value="square",  label="outer shape")
            co_ = gr.Dropdown(COLOR_NAMES,  value="green",   label="outer colour")
            si  = gr.Dropdown(SHAPES,       value="circle",  label="inner shape")
            ci  = gr.Dropdown(COLOR_NAMES,  value="yellow",  label="inner colour")
        with gr.Row():
            bg2 = gr.Dropdown(BG_NAMES,      value="beige",  label="background")
            cp2 = gr.Dropdown(COMPOSITIONS, value="nested",  label="composition")
            sd2 = gr.Slider(0, 9999, value=0, step=1,        label="seed")
        btn2 = gr.Button("Generate", variant="primary")
        out2 = gr.Image(label="generated icon", show_download_button=True, height=320)
        btn2.click(tab_compose, [so, co_, si, ci, bg2, cp2, sd2], out2)
        gr.Examples(
            [
                ["square",   "green",  "circle",   "yellow",  "beige",     "nested", 1],
                ["hexagon",  "cyan",   "triangle", "magenta", "navy",      "nested", 2],
                ["pentagon", "purple", "star",     "orange",  "lightgrey", "nested", 3],
                ["star",     "blue",   "star",     "blue",    "black",     "single", 4],
                ["circle",   "red",    "circle",   "red",     "white",     "single", 5],
            ],
            inputs=[so, co_, si, ci, bg2, cp2, sd2], label="Example compositions",
        )

    # ───── Tab 3 ─────
    with gr.Tab("🎨 3. Style Transfer"):
        gr.Markdown("**AdaIN** mixes the *channel statistics* of a *style* image into a *content* image. "
                    "α controls strength: 0 = pure content, 1 = full style swap.")
        with gr.Row():
            with gr.Column():
                gr.Markdown("**Content icon**")
                csh = gr.Dropdown(SHAPES,      value="circle", label="shape")
                cco = gr.Dropdown(COLOR_NAMES, value="red",    label="colour")
                cbg = gr.Dropdown(BG_NAMES,    value="white",  label="background")
            with gr.Column():
                gr.Markdown("**Style icon**")
                ssh = gr.Dropdown(SHAPES,      value="square", label="shape")
                sco = gr.Dropdown(COLOR_NAMES, value="blue",   label="colour")
                sbg = gr.Dropdown(BG_NAMES,    value="navy",   label="background")
        alpha = gr.Slider(0.0, 1.0, value=0.7, step=0.05, label="α — style strength")
        btn3 = gr.Button("Run AdaIN", variant="primary")
        with gr.Row():
            i_c = gr.Image(label="content",       show_download_button=True, height=240)
            i_s = gr.Image(label="style",         show_download_button=True, height=240)
            i_o = gr.Image(label="content × style", show_download_button=True, height=240)
        btn3.click(tab_style, [csh, cco, cbg, ssh, sco, sbg, alpha], [i_c, i_s, i_o])

    # ───── Tab 4 ─────
    with gr.Tab("🔥 4. Cross-Attention"):
        gr.Markdown("Attention weights at the 16×16 decoder bottleneck, averaged over heads, "
                    "shown per condition token.")
        with gr.Row():
            so4 = gr.Dropdown(SHAPES,      value="hexagon",  label="outer shape")
            co4 = gr.Dropdown(COLOR_NAMES, value="cyan",     label="outer colour")
            si4 = gr.Dropdown(SHAPES,      value="triangle", label="inner shape")
            ci4 = gr.Dropdown(COLOR_NAMES, value="magenta",  label="inner colour")
        with gr.Row():
            bg4 = gr.Dropdown(BG_NAMES,      value="navy",   label="background")
            cp4 = gr.Dropdown(COMPOSITIONS, value="nested",  label="composition")
            sd4 = gr.Slider(0, 9999, value=2, step=1,        label="seed")
        btn4 = gr.Button("Visualise attention", variant="primary")
        gallery4 = gr.Gallery(label="generated + per-token attention", columns=7, height=240, show_download_button=True)
        btn4.click(tab_attention, [so4, co4, si4, ci4, bg4, cp4, sd4], gallery4)

    # ───── Tab 5 ─────
    with gr.Tab("🧭 5. Latent Explorer"):
        gr.Markdown("Slide the first 8 latent dimensions to see how they affect the generated icon.")
        with gr.Row():
            so5 = gr.Dropdown(SHAPES,      value="circle", label="shape")
            co5 = gr.Dropdown(COLOR_NAMES, value="red",    label="colour")
            bg5 = gr.Dropdown(BG_NAMES,    value="white",  label="background")
        sliders = [gr.Slider(-3, 3, value=0.0, step=0.1, label=f"z[{i}]") for i in range(8)]
        btn5 = gr.Button("Render", variant="primary")
        out5 = gr.Image(label="generated icon", show_download_button=True, height=320)
        btn5.click(tab_latent, [so5, co5, bg5, *sliders], out5)

    # ───── Tab 6 ─────
    with gr.Tab("🖼️ 6. Gallery"):
        gr.Markdown("Render a fresh **8 × 8** gallery of icons every click.")
        seed6 = gr.Slider(0, 9999, value=0, step=1, label="seed")
        btn6  = gr.Button("Generate gallery", variant="primary")
        out6  = gr.Image(label="64 icons", show_download_button=True, height=520)
        btn6.click(tab_gallery, seed6, out6)
        gr.Examples([[3], [11], [42], [314], [2718]], inputs=seed6, label="Example seeds")

    gr.Markdown(
        "---\n"
        "© 2026 **Mohamed El-sadek** · MIT License · "
        "[GitHub](https://github.com/Mohameddfxxcxx/CrossAttention-IconFusion-AI) · "
        "[Kaggle](https://www.kaggle.com/code/mohamedelsadek44/geometric-icons-cond-vae)"
    )

# ──────────────── batch-mode smoke test ────────────────
is_batch = (
    os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "").lower() == "batch"
    or os.environ.get("CI") == "true"
)
FORCE_LAUNCH = False

print("Smoke-testing all six Gradio tab handlers ...")
_img1, _cap = tab_random(0);                                                        print("  tab 1 (random)    OK   ->", _cap)
_img2       = tab_compose("square","green","circle","yellow","beige","nested",1);   print("  tab 2 (compose)   OK")
_ic,_is,_io = tab_style("circle","red","white","square","blue","navy",0.7);         print("  tab 3 (style)     OK")
_panels     = tab_attention("hexagon","cyan","triangle","magenta","navy","nested",2); print("  tab 4 (attention) OK   ->", len(_panels), "panels")
_img5       = tab_latent("circle","red","white",0,0,0,0,0,0,0,0);                   print("  tab 5 (latent)    OK")
_img6       = tab_gallery(3);                                                       print("  tab 6 (gallery)   OK   ->", _img6.size)

_img1.save(DIRS["app"] / "demo_random.png")
_img6.save(DIRS["app"] / "demo_gallery.png")
print("\nApp artefacts written to", DIRS['app'])

if FORCE_LAUNCH and not is_batch:
    demo.launch(share=True, debug=False, inline=False)
else:
    print("\nBatch / non-interactive run detected — skipping demo.launch() to avoid blocking.")
    print("To launch the interactive UI in Colab/Jupyter, set FORCE_LAUNCH=True and re-run this cell.")

## 41. Final Results Summary

**Deliverables produced by this notebook**

| File | Description |
|---|---|
| `checkpoints/vae_best.pt` | Best-validation Cond-VAE checkpoint |
| `checkpoints/vae_final.pt` | Final-epoch checkpoint |
| `samples/final_samples.png` | The five required final samples |
| `samples/gallery_8x8.png` | 64-image generated gallery |
| `figures/training_curves.png` | Loss / KL / Recon curves |
| `figures/style_transfer.png` | AdaIN content × style sweep |
| `figures/attention_map_*.png` | Cross-attention overlays |
| `figures/error_analysis.png` | Worst-8 reconstructions |
| `figures/latent_space.png` | PCA + t-SNE of test latents |
| `results/metrics.json` | Quantitative metrics on the test set |
| `results/ablation.json` | Cross-attention vs. no-attention CCA |
| `results/history.json` | Per-epoch training history |

**Bonus tasks completed**

- ✅ Cross-attention between shape / icon tokens and feature maps (§18)
- ✅ Style transfer between icons via AdaIN (§19, §31)
- ✅ Interactive Gradio app with 6 tabs (§40)
